In [1]:
import os
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
import numpy as np 
import cv2
import matplotlib.pyplot as plt 
from tensorflow.keras.layers import Layer, Flatten, Dense, Dropout
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.callbacks import ModelCheckpoint

In [2]:
train_directory = "/Users/krudantrandai/Style Transfer/Train"
test_directory = "/Users/krudantrandai/Style Transfer/Test"
validation_directory = "/Users/krudantrandai/Style Transfer/validation"

In [17]:
img_size = (224, 224)
batch_size = 32
num_classes = 1  # Adjust based on the number of classes in your dataset

In [18]:
train_datagen = ImageDataGenerator(rescale=1./255,validation_split=0.2)
validation_datagen = ImageDataGenerator(rescale=1./255,validation_split=0.2)
test_datagen = ImageDataGenerator(rescale=1./255,validation_split=0.2)

In [19]:
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.models import Model


input_layer = Input(shape=(224, 224, 3))

# Block 1
x = Conv2D(64, (3, 3), activation='relu', padding='same', name='block1_conv1')(input_layer)
x = Conv2D(64, (3, 3), activation='relu', padding='same', name='block1_conv2')(x)
x = MaxPooling2D((2, 2), strides=(2, 2), name='block1_pool')(x)

# Block 2
x = Conv2D(128, (3, 3), activation='relu', padding='same', name='block2_conv1')(x)
x = Conv2D(128, (3, 3), activation='relu', padding='same', name='block2_conv2')(x)
x = MaxPooling2D((2, 2), strides=(2, 2), name='block2_pool')(x)

# Block 3
x = Conv2D(256, (3, 3), activation='relu', padding='same', name='block3_conv1')(x)
x = Conv2D(256, (3, 3), activation='relu', padding='same', name='block3_conv2')(x)
x = Conv2D(256, (3, 3), activation='relu', padding='same', name='block3_conv3')(x)
x = Conv2D(256, (3, 3), activation='relu', padding='same', name='block3_conv4')(x)
x = MaxPooling2D((2, 2), strides=(2, 2), name='block3_pool')(x)

# Block 4
x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block4_conv1')(x)
x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block4_conv2')(x)
x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block4_conv3')(x)
x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block4_conv4')(x)
x = MaxPooling2D((2, 2), strides=(2, 2), name='block4_pool')(x)

# Block 5
x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block5_conv1')(x)
x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block5_conv2')(x)
x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block5_conv3')(x)
x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block5_conv4')(x)
x = MaxPooling2D((2, 2), strides=(2, 2), name='block5_pool')(x)

# Flatten and fully connected layers
WT = Flatten(name='flatten')(x)
x = Dense(4096, activation='relu', name='fc1')(WT)
x = Dense(4096, activation='relu', name='fc2')(x)
output_layer = Dense(1, activation='sigmoid', name='predictions')(x)

model = Model(inputs = input_layer, outputs = output_layer)

In [20]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


In [21]:
train_generator = train_datagen.flow_from_directory(
    train_directory,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='training'
)

validation_generator = validation_datagen.flow_from_directory(
    validation_directory,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='validation'
)


Found 674 images belonging to 2 classes.
Found 56 images belonging to 2 classes.


In [22]:
checkpoint_path = "Models VGG/VGG_19_MS.h5"
cp_callback = ModelCheckpoint(checkpoint_path, save_weight_only = False,
                              save_best_only = True,verbose =1)
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // batch_size,
    epochs=50,  
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // batch_size,
    callbacks = [cp_callback]
)


Epoch 1/50
21/21 [==============================] - ETA: 0s - loss: 117.3292 - accuracy: 0.5016
Epoch 1: val_loss improved from inf to 0.69215, saving model to Models VGG/VGG_19_MS.h5


/Users/krudantrandai/tensorflow/lib/python3.10/site-packages/keras/src/engine/training.py:3079: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


21/21 [==============================] - 74s 3s/step - loss: 117.3292 - accuracy: 0.5016 - val_loss: 0.6921 - val_accuracy: 0.5312
Epoch 2/50
21/21 [==============================] - ETA: 0s - loss: 0.7475 - accuracy: 0.5000
Epoch 2: val_loss improved from 0.69215 to 0.68834, saving model to Models VGG/VGG_19_MS.h5
21/21 [==============================] - 85s 4s/step - loss: 0.7475 - accuracy: 0.5000 - val_loss: 0.6883 - val_accuracy: 0.5938
Epoch 3/50
21/21 [==============================] - ETA: 0s - loss: 0.6953 - accuracy: 0.4984
Epoch 3: val_loss did not improve from 0.68834
21/21 [==============================] - 78s 4s/step - loss: 0.6953 - accuracy: 0.4984 - val_loss: 0.6924 - val_accuracy: 0.5312
Epoch 4/50
21/21 [==============================] - ETA: 0s - loss: 0.7004 - accuracy: 0.5000
Epoch 4: val_loss did not improve from 0.68834
21/21 [==============================] - 77s 4s/step - loss: 0.7004 - accuracy: 0.5000 - val_loss: 0.6933 - val_accuracy: 0.4688
Epoch 5/50
21/

21/21 [==============================] - 81s 4s/step - loss: 0.6934 - accuracy: 0.4844 - val_loss: 0.6936 - val_accuracy: 0.4688
Epoch 32/50
21/21 [==============================] - ETA: 0s - loss: 0.6936 - accuracy: 0.4984
Epoch 32: val_loss did not improve from 0.68557
21/21 [==============================] - 95s 5s/step - loss: 0.6936 - accuracy: 0.4984 - val_loss: 0.6934 - val_accuracy: 0.5000
Epoch 33/50
21/21 [==============================] - ETA: 0s - loss: 0.6937 - accuracy: 0.5016
Epoch 33: val_loss did not improve from 0.68557
21/21 [==============================] - 83s 4s/step - loss: 0.6937 - accuracy: 0.5016 - val_loss: 0.6930 - val_accuracy: 0.5312
Epoch 34/50
21/21 [==============================] - ETA: 0s - loss: 0.6935 - accuracy: 0.5016
Epoch 34: val_loss did not improve from 0.68557
21/21 [==============================] - 79s 4s/step - loss: 0.6935 - accuracy: 0.5016 - val_loss: 0.6933 - val_accuracy: 0.5000
Epoch 35/50
21/21 [==============================] - ET